In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
from scipy import signal

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 11 &mdash; The Discrete Fourier Transform</h1></center>

Every spectrum computed in this course so far came from `np.fft.fft`, and Module 5 asked you to take that on trust. Here we open the box.
Starting from the Fourier series of Module 4, we replace continuous time by a finite sequence of $N$ samples and find that the transform
becomes a sum, then a matrix&ndash;vector product (the operator view of Module 8), and finally the fast algorithm used everywhere in
geophysics. Because a finite record is implicitly periodic, the DFT has its own rules: convolution wraps around, the frequency spacing is
fixed by the record length, and a sinusoid that does not complete a whole number of cycles leaks energy into every bin. Knowing these
rules is the difference between reading a seismic spectrum and being misled by it.

# Outline
- [11.1 From Fourier Series to Discrete Sequences](#11.1-From-Fourier-Series-to-Discrete-Sequences)
- [11.2 The Inverse Discrete Fourier Series](#11.2-The-Inverse-Discrete-Fourier-Series)
- [11.3 The Forward Discrete Fourier Series](#11.3-The-Forward-Discrete-Fourier-Series)
  - [11.3.1 Example 1: a periodic boxcar](#11.3.1-Example-1:-a-periodic-boxcar)
- [11.4 Compact Notation: $W_N$](#11.4-Compact-Notation:-$W_N$)
  - [11.4.1 Example 2: a sampled cosine](#11.4.1-Example-2:-a-sampled-cosine)
- [11.5 The Discrete Fourier Series as a Matrix](#11.5-The-Discrete-Fourier-Series-as-a-Matrix)
- [11.6 Properties of the Discrete Fourier Series](#11.6-Properties-of-the-Discrete-Fourier-Series)
- [11.7 The Discrete Fourier Transform](#11.7-The-Discrete-Fourier-Transform)
  - [11.7.1 Example 3: four standard DFT pairs](#11.7.1-Example-3:-four-standard-DFT-pairs)
  - [11.7.2 Example 4: an inverse DFT](#11.7.2-Example-4:-an-inverse-DFT)
  - [11.7.3 Example 5: a squared cosine](#11.7.3-Example-5:-a-squared-cosine)
- [11.8 The DFT as a Matrix](#11.8-The-DFT-as-a-Matrix)
- [11.9 Properties of the DFT](#11.9-Properties-of-the-DFT)
- [11.10 Linear Convolution with the DFT](#11.10-Linear-Convolution-with-the-DFT)
- [11.11 Fast Fourier Transforms](#11.11-Fast-Fourier-Transforms)
- [11.12 The Frequency Axis, Resolution and Leakage](#11.12-The-Frequency-Axis,-Resolution-and-Leakage)
- [11.13 Parting Thoughts](#11.13-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="11.1"></a>
## 11.1 From Fourier Series to Discrete Sequences

Now that we have covered the digital sampling theorem, we are better equipped to understand what is going on "under the hood" when we have been applying the (discrete or fast) Fourier Transform.  

You'll recall that when we looked at **continuous** functions:

1. The Fourier Series of a **periodic continuous** function has a **discrete** Fourier representation ($k\Delta \omega$);

2. The Fourier Transform of a **continuous** function has a **continuous** Fourier representation ($\omega$)

We'll now look at two examples of **discrete** sequences and observe that:

3. The **discrete** Fourier Series (DFS) of a **periodic discrete** sequence has a **periodic discrete** Fourier representation.

4. The **discrete** Fourier Transform (DFT) of a **discrete** sequence has a **discrete** Fourier representation.

We will start first with the DFS, which represents an extension of the continuous-time Fourier Series.  This will motivate our analysis of the DFT.  We will then look at the DFTs of a number of short filters.

The **discrete Fourier Series** (DFS) is related to a **continuous Fourier Series** (CFS) in that it is a repetitive signal with a fixed interval of repetition.  However, whereas a time-series CFS has a periodicity of duration $T$ seconds, a DFS is a sequence of numbers with a periodicity every $N$ values or samples. 

Let $\tilde{x}[n]$ be a **periodic sequence** with period $N$,

$$ \tilde{x}[n] = \tilde{x}[n+N] \tag{1}$$

where $n=[\dots,-2,-1,0,1,2,\dots]$, and we use a tilde symbol to recognize that the time series is **periodic**. 
This function can thus describe an infinitely repeating sequence via the following:

$$ \tilde{x}[n] = \sum_{k=-\infty}^{\infty} x[n-Nk] \tag{2}$$

where $x[n]$ represents a **single period** of the sequence with index ranging from $n=0,N-1$.

In [ ]:
C = DSP_COLORS
n = np.arange(5); xn = np.array([2, 4, 6, 8, 10])
nt = np.arange(-10, 15); xt = np.tile(xn, 5)              # five periods, n = -10 ... 14
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, idx, v, ttl in ((a1, n, xn, '(a) One period, $x[n]$'), (a2, nt, xt, r'(b) Periodic extension $\tilde{x}[n]$, period $N = 5$')):
    ml, sl, bl = ax.stem(idx, v, basefmt='k-')
    plt.setp(sl, color=C['sampled']); plt.setp(ml, color=C['sampled'], ms=6)
    ax.set_title(ttl); ax.set_ylabel('amplitude'); ax.set_xlim(-10.5, 14.5); ax.set_ylim(-0.5, 11.5)
for p in range(-2, 4):
    a2.axvline(5*p - 0.5, ls=':', color=C['neutral'], lw=1)
a2.set_xlabel('sample index $n$')
plt.tight_layout(); plt.show()

<b>Figure 11.1.</b> The base sequence and its periodic extension. (a) One period, $x[n] = [2, 4, 6, 8, 10]$. (b) The periodic sequence $\tilde{x}[n]$ formed by repeating (a) indefinitely; dotted lines mark the period boundaries, $N = 5$. <i>(Python-generated.)</i>

Although strictly speaking $\tilde{x}[n]$ does not have a Fourier Transform because it is not absolutely summable (and therefore has an infinite amount of energy), it can be expressed in terms of a discrete Fourier series over a single period (i.e., $k=0,N-1$):

<div class="alert alert-info" role="alert">
$$ \tilde{x}[n] = \frac{1}{N} \sum_{k=0}^{N-1} \tilde{X}[k]e^{i2\pi k n/N}, \quad n=-\infty,\infty\tag{3}$$
</div>

where $n$ is the "time" variable and $k$ is the "frequency" variable. Note that whereas the index $\tilde{X}[k]$ ranges $k=0,N-1$, the index on $\tilde{x}[n]$ ranges $n=-\infty,\infty$.

<a name="11.2"></a>
## 11.2 The Inverse Discrete Fourier Series

To make the connection to what you have learned earlier in the course on the (continuous) Fourier Series, you might recall that the **reconstruction** of periodic function $x(t)$ was given by:

$$ x(t) = \sum_{k=-\infty}^{\infty} C_k {\rm e}^{i k \omega_0 t} \tag{4}$$

In this case we are constructing the **periodic continuous** function $x(t)$ by summing over a potentially infinite number of coefficients $C_k$ weighted by the Fourier kernel ${\rm e}^{i k \omega_0 t}$ where $t$ is a continuous variable. 

Let's start the connection between the discrete and continuous Fourier Series by replacing the fundamental angular frequency in equation (4) with $\omega_0 = \frac{2\pi}{T}$ yields:

$$ x(t) = \sum_{k=-\infty}^{\infty} C_k {\rm e}^{i 2\pi k t/T} \tag{5}$$

The final substitutions leading to equation (3) are: 

* (1) replacing $\frac{t}{T}\rightarrow\frac{n}{N}$; 
$$ x(t) = \sum_{k=-\infty}^{\infty} C_k {\rm e}^{i 2\pi k n/N} \tag{6}$$
* (2) recognizing that we only need to consider a single period (i.e., $0\le n\le N-1$) because our frequency domain response will be periodic with period $N$; 
$$ x(t) = \sum_{k=0}^{N-1} C_k {\rm e}^{i 2\pi k n/N} \tag{7}$$
* (3) replacing continuous function $x(t)$ with discrete sequence $\tilde{x}[n]$ and exchanging our potentially infinite number of discrete $C_k$ coefficients (which have an implicit $\frac{1}{T}$ weighting) with a similarly discrete $N$-point sequence $\tilde{X}[k]$ (that has an explicit $\frac{1}{N}$ weighting).  $$ \tilde{x}[n] = \frac{1}{N}\sum_{k=0}^{N-1}  \tilde{X}[k]{\rm e}^{i 2\pi k n/N} \tag{8}$$


The last point is a consequence of only needing to represent the signal at discrete frequencies up to the Nyquist frequency (unlike a continuous signal that requires summation over an infinite number of coefficients for an exact representation).  Essentially, this is a decomposition of $\tilde{x}[n]$ into a sum of $N$ harmonically related exponentials weighted by their Fourier coefficients $\tilde{X}[k]$.

<a name="11.3"></a>
## 11.3 The Forward Discrete Fourier Series

To specify the forward DFS transform, one may multiply both sides of equation (8) by 
$e^{-i2\pi l n/N}$ and sum over a full period:

$$ \tilde{x}[n] e^{-i2\pi l n/N} = \frac{1}{N} \sum_{k=0}^{N-1} \tilde{X}[k]e^{i2\pi (k-l) n/N} \tag{9} $$

We can use the fact that complex exponentials are orthogonal 

$$\sum_{n=0}^{N-1} e^{i2\pi (k-l) n/N}
=
N \delta[k-l]
=
\left\{
\begin{array}{c}
N, \quad k=l \\
0, \quad k\ne l
\end{array}
\right.{}
\tag{10}
$$

to simplify this expression.

<div class="alert alert-success">
<h3><center>Distraction A: Why complex exponentials are orthogonal</center></h3>

To demonstrate this let's examine the case where $N=4$:

$$
\sum_{n=0}^{N-1} e^{i2\pi (k-l) n/N}
= 
\sum_{n=0}^{3} e^{i2\pi (k-l) n/4}
=
e^{0}+e^{i\pi (k-l) /2}+e^{i\pi (k-l)}+e^{i3\pi (k-l)/2}
=
1+e^{i\pi (k-l)/2}+e^{i\pi (k-l)}+e^{i3\pi (k-l)/2} \tag{11}
$$

Now if $k=l$ then the following is true:

$$\sum_{n=0}^{3} e^{i2\pi (k-l) n/4}=\left.1+e^{i\pi (k-l)/2}+e^{i\pi (k-l)}+e^{i3\pi (k-l)/2}\right|_{k=l} = 1 + 1 + 1 + 1=4=N \tag{12}$$

Now if, say, $k=l+1$ then the following is true:

$$\sum_{n=0}^{3} e^{i2\pi (k-l) n/4}=\left.1+e^{i\pi (k-l)/2}+e^{i\pi (k-l)}+e^{i3\pi (k-l)/2}\right|_{k=l+1} = 1+e^{i\pi/2}+e^{i\pi}+e^{i3\pi/2}=1+i-1-i=0 \tag{13}$$

One can show this to be generally true for any value of $N$ and $k\neq l$.
</div>

Let's now sum equation (9) in range $n=0,N-1$ to extract $\tilde{X}[k]$ for every $k$ value.

$$ 
\begin{eqnarray}
\sum_{n=0}^{N-1} \tilde{x}[n] e^{-i2\pi l n/N} 
&=& \frac{1}{N} \sum_{n=0}^{N-1} \sum_{k=0}^{N-1} \tilde{X}[k]e^{i2\pi (k-l) n/N} \tag{14}\\
&=& \frac{1}{N} \sum_{k=0}^{N-1} \tilde{X}[k] \left(\sum_{n=0}^{N-1}  e^{i2\pi (k-l) n/N}\right) \tag{15}\\
&=& \frac{1}{N} \sum_{k=0}^{N-1} \tilde{X}[k] N \delta[k-l] \tag{16} \\
&=&  \tilde{X}[l] \tag{17} 
\end{eqnarray}
$$

Switching back from dummy index $l$ to $k$ on the left hand side of equation (14) and the right hand side of equation (17), the above result shows that the Fourier coefficients of the DFS transform are given by:

<div class="alert alert-info" role="alert">
$$
\tilde{X}[k] = \sum_{n=0}^{N-1} \tilde{x}[n] e^{-i2\pi k n/N}, \quad k=-\infty,\infty \tag{18}
$$
</div>

where the index of $\tilde{X}[k]$ ranges between $k=-\infty,\infty$ because this is an infinitely repeating sequence.  Note that the DFS coefficients are similarly periodic with period N:

$$\tilde{X}[k]=\tilde{X}[k+N], \quad k\in I \tag{19} $$

Based on the equations above we write

$$\tilde{x}[n] \overset{DFS}{\Longleftrightarrow}\tilde{X}[k] \tag{20}$$

to illustrate the duality of the discrete Fourier Series.

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as a basis.</b> The $N$ numbers $W_N^k$ (introduced in Section 11.4) are the $N$-th roots of unity from Module 3
(Section 3.6): equally spaced points on the unit circle. Orthogonality, equation (10), is a geometric series (MATH 213) whose terms are those points,
and they sum to zero unless $k = l$. It is the discrete twin of the orthogonality of $e^{ik\omega_0 t}$ over one period in Module 4, which let us pick off
each Fourier-series coefficient.
</div>

<a name="11.3.1"></a>
### 11.3.1 Example 1: a periodic boxcar

**Q:** Let us find a discrete Fourier series representation for the periodic sequence

$$ \tilde{x}[n] = \sum_{k=-\infty}^{\infty} x[n-10k],\quad n=-\infty,\infty \tag{21}$$

where $N=10$ and our base sequence $x[n]$ is given by:

$$ x[n] = 
\left\{
\begin{array}{cc}
1, & 0 \le n < 5 \\
0, & 5 \le n < 10 \\
\end{array}
\right.{}
\tag{22}
$$

**A:** Note that $\tilde{x}[n]$ is periodic with a period $N=10$.  Therefore, the DFS coefficients are given by equation (18):

$$
\tilde{X}[k] = \sum_{n=0}^{9} \tilde{x}[n] e^{-i2\pi k n/10}  = \sum_{n=0}^{9} \tilde{x}[n] e^{-i\pi k n/5} \tag{23} 
$$

Substituting in $x[n]$ for $\tilde{x}[n]$ (because they are equivalent over the full period N  between $n=0,9$) yields:

$$
\tilde{X}[k] = \sum_{n=0}^{9} x[n] e^{-i\pi k n/5} = \sum_{n=0}^{4} e^{-i\pi k n/5}  \tag{24}
$$

where in equation (24) we used the fact that $x[n]=1$ between $n=0,4$ and is zero otherwise.  Recognizing that we can pull out a power of $^n$, we can write:

$$
\tilde{X}[k] = \sum_{n=0}^{4} \left(e^{-i\pi k/5}\right)^n \tag{25}
$$

which is a [geometric series](https://en.wikipedia.org/wiki/Geometric_series) where $r=e^{-i\pi k/5}$.  A geometric series has the following sum:

$$S_n = \sum_{n=0}^{N-1} r^n = \frac{1-r^N}{1-r} \tag{26}$$

which allows us to write

$$
\tilde{X}[k] = \sum_{n=0}^{4} \left(e^{-i\pi k/5}\right)^n = \frac{1-e^{-i\pi k}}{1-e^{-i\pi k/5}} 
=
\left\{
\begin{array}{cc}
5 & k=0\\
\frac{2}{1-e^{-i\pi k/5}} & k\,\mathrm{odd} \\
0 & k\,\mathrm{even} \\
\end{array}
\right.{}, \quad k=-\infty,\infty \tag{27}
$$

The values for $k=0$ (or $kN$) can be found by inspecting the middle term, which in this case turns out to be the summation over $x[n]$ coefficients (i.e., 5 ones).  The DFS for all other values of $k$ may be found according to equation (27).  Here are the coefficients, as well as a graphical representation of their real and imaginary components.

Let's look at the answer graphically for a single period:

In [ ]:
C = DSP_COLORS
def stem_ri(X, xlabel='frequency index $k$', ylabel='coefficient', what='$X[k]$', ylim=None, idx=None):
    """Real (a) and imaginary (b) parts of a sequence as stem plots (used for every example in this module)."""
    idx = np.arange(len(X)) if idx is None else idx
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
    for ax, v, ttl in zip(axs, (np.real(X), np.imag(X)), ('(a) Real part of ', '(b) Imaginary part of ')):
        ml, sl, bl = ax.stem(idx, v, basefmt='k-')
        plt.setp(sl, color=C['freq_domain']); plt.setp(ml, color=C['freq_domain'], ms=6)
        ax.set_title(ttl + what); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(*ylim)
    plt.tight_layout(); plt.show()

x = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
X = np.fft.fft(x); print(np.round(X, 4))
stem_ri(X, what=r'$\tilde{X}[k]$', ylim=(-1.6, 5.4))

<b>Figure 11.2.</b> The real (a) and imaginary (b) parts of the DFS coefficients of Example 1, equation (27): $\tilde{X}[0] = 5$, the even coefficients vanish, and the odd ones are $2/(1 - e^{-i\pi k/5})$. The printed array above the plot lists the same values. <i>(Python-generated.)</i>

<a name="11.4"></a>
## 11.4 Compact Notation: $W_N$

To simplify notation one often sees the $W_N$ notation that uses the following definition:

<div class="alert alert-info" role="alert">
$$W_N = e^{-i2\pi/N} \tag{28}$$
</div>

Thus, we can write:

$$\left(W_N\right)^{kn} = W_N^{kn} = e^{-i2\pi kn/N} \tag{29}$$

<div class="alert alert-info" role="alert">
Using this notation the forward and inverse DFS operations may be written:

$$
\tilde{X}[k]= \sum_{n=0}^{N-1} \tilde{x}[n] e^{-i2\pi nk/N} = \sum_{n=0}^{N-1} \tilde{x}[n] W_N^{nk}, \quad k=-\infty,\infty \tag{30}
$$

and

$$
 \tilde{x}[n] = \frac{1}{N}\sum_{k=0}^{N-1} \tilde{X}[k]  e^{i2\pi nk/N} = \frac{1}{N}\sum_{k=0}^{N-1} \tilde{X}[k]  W_N^{-nk}, \quad n=-\infty,\infty \tag{31}
$$

respectively.
</div>

<b>Exercises</b>

If $W_N = e^{-i2\pi/N}$, what do the following equal?

* $W_4^{0}$

* $W_4^{2}$

* $W_4^{4p}$ where $p$ is integer.

* $W_4^{4p+2}$ where $p$ is integer.

<details>
<summary><b>Show solutions &mdash; Exercises</b></summary>

If $W_N = e^{-i2\pi/N}$, what do the following equal?

<ul><li>$W_4^{0}$</li></ul>

$W_4^{0} = e^{-i2\pi 0/N}=e^{-i2\pi 0/4}=e^{-i 0}=1$. 

<ul><li>$W_4^{2}$</li></ul>

$W_4^{2} = e^{-i2\pi 2/N}=e^{-i2\pi 2/4}=e^{-i\pi }=-1$

<ul><li>$W_4^{4p}$ where $p$ is integer.</li></ul>


$W_4^{4p} = e^{-i2\pi 4p/N}=e^{-i2\pi 4p /4}=e^{-i 2\pi p}=1$ for all p integers.


<ul><li>$W_4^{4p+2}$ where $p$ is integer.</li></ul>


$W_4^{4p+2} = e^{-i2\pi (4p+2)/N}=e^{-i2\pi (4p+2) /4}=e^{-i 2\pi p - i2\pi/2}=-e^{i 2\pi p}=-1$ for all p integers.
</details>

<a name="11.4.1"></a>
### 11.4.1 Example 2: a sampled cosine

**Q:** Find the DFS expansion of the sequence

$$\tilde{x}[n] = A \,\mathrm{cos}\left(\frac{n\pi}{2}\right), \quad n=-\infty,\infty \tag{32}$$

**A:** Because $\tilde{x}[n]$ is periodic with $N=4$ (i.e., $\tilde{x}[n]=[A,0,-A,0]$), the DFS coefficients may be found by evaluating the sum

$$
\tilde{X}[k] = \sum_{n=0}^{3} \tilde{x}[n] W_4^{nk}, \quad k=-\infty,\infty \tag{33}
$$

Let's look at the different $n$ values substituting in the values of $\tilde{x}[n]$

$$
\begin{eqnarray}
\tilde{X}[0] &=& A W_4^{0*0} -  A W_4^{2*0} &=& A W_4^{0} -  A W_4^{0} & = 0  \\
\tilde{X}[1] &=& A W_4^{0*1} -  A W_4^{2*1} &=& A W_4^{0} -  A W_4^{2} & = 2A \\
\tilde{X}[2] &=& A W_4^{0*2} -  A W_4^{2*2} &=& A W_4^{0} -  A W_4^{4} & = 0  \\
\tilde{X}[3] &=& A W_4^{0*3} -  A W_4^{2*3} &=& A W_4^{0} -  A W_4^{6} & = 2A \\
\end{eqnarray}
\tag{34}
$$

where we have used $W_4^{0} = e^{-i2\pi 0/4}=1$ and $W_4^{2} = e^{-i \pi}=- 1$, as well as
for any integer $p$: $W_4^{4p} =  e^{-i2\pi p}=1$ and $W_4^{4p+2} =  e^{-i2\pi p - i\pi}=-1$.  Below is a printout of the coefficients, as well as stem graphs of the real and imaginary components. Let's look at these graphically for a single period assuming $A=1$:

In [ ]:
x1 = np.array([1, 0, -1, 0])                 # A = 1
X1 = np.fft.fft(x1); print(np.round(X1, 6))
stem_ri(X1, what=r'$\tilde{X}[k]$', ylim=(-0.5, 2.5))

<b>Figure 11.3.</b> The real (a) and imaginary (b) parts of the DFS coefficients of Example 2, equation (34), for $A = 1$: $\tilde{X} = [0, 2, 0, 2]$. <i>(Python-generated.)</i>

<a name="11.5"></a>
## 11.5 The Discrete Fourier Series as a Matrix

By inspecting the above example, you might notice that the operations can be written using **matrix notation** (i.e., $\left[W_N^{nk}\right]$).  Let's do just that here:

$$
\begin{eqnarray}
\left(
\begin{array}{c}
\tilde{X}[0] \\
\tilde{X}[1]\\
\tilde{X}[2]\\
\tilde{X}[3]\\
\end{array}
\right)
&=&
\left(
\begin{array}{c}
W_4^{0*0} & W_4^{1*0} & W_4^{2*0} & W_4^{3*0}\\
W_4^{0*1} & W_4^{1*1} & W_4^{2*1} & W_4^{3*1}\\
W_4^{0*2} & W_4^{1*2} & W_4^{2*2} & W_4^{3*2}\\
W_4^{0*3} & W_4^{1*3} & W_4^{2*3} & W_4^{3*3}\\
\end{array}
\right)
\left(
\begin{array}{c}
\tilde{x}[0] \\
\tilde{x}[1] \\
\tilde{x}[2] \\
\tilde{x}[3] \\
\end{array}
\right) 
=
\left(
\begin{array}{c}
W_4^{0} & W_4^{0} & W_4^{0} & W_4^{0}\\
W_4^{0} & W_4^{1} & W_4^{2} & W_4^{3}\\
W_4^{0} & W_4^{2} & W_4^{4} & W_4^{6}\\
W_4^{0} & W_4^{3} & W_4^{6} & W_4^{9}\\
\end{array}
\right)
\left(
\begin{array}{c}
\tilde{x}[0] \\
\tilde{x}[1] \\
\tilde{x}[2] \\
\tilde{x}[3] \\
\end{array}
\right) 
\\
\,
&
=
&
\left(
\begin{array}{cccc}
 1 & 1 & 1 & 1 \\
 1 & e^{\frac{-i \pi }{2}} & e^{-i \pi } & e^{\frac{-3 i \pi }{2}} \\
 1 & e^{-i \pi } & e^{-2 i \pi } & e^{-3 i \pi } \\
 1 & e^{\frac{-3 i \pi }{2}} & e^{-3 i \pi } & e^{\frac{-9 i \pi }{2}} \\
\end{array}
\right)
\left(
\begin{array}{c}
\tilde{x}[0] \\
\tilde{x}[1] \\
\tilde{x}[2] \\
\tilde{x}[3] \\
\end{array}
\right) 
=
\left(
\begin{array}{cccc}
 1 & 1 & 1 & 1 \\
 1 & -i & -1 & i \\
 1 & -1 & 1 & -1 \\
 1 & i & -1 & -i \\
\end{array}
\right)
\left(
\begin{array}{c}
\tilde{x}[0] \\
\tilde{x}[1] \\
\tilde{x}[2] \\
\tilde{x}[3] \\
\end{array}
\right) \\
\end{eqnarray}\tag{35}
$$

Inserting the input values $[A,0,-A,0]$ yields:

$$
\left(
\begin{array}{c}
\tilde{X}[0] \\
\tilde{X}[1]\\
\tilde{X}[2]\\
\tilde{X}[3]\\
\end{array}
\right)
=
\left(
\begin{array}{cccc}
 1 & 1 & 1 & 1 \\
 1 & -i & -1 & i \\
 1 & -1 & 1 & -1 \\
 1 & i & -1 & -i \\
\end{array}
\right)
\left(
\begin{array}{c}
A \\
0 \\
-A \\
0 \\
\end{array}
\right) 
=
\left(
\begin{array}{c}
0 \\
2A \\
0 \\
2A \\
\end{array}
\right) \tag{36}
$$

which yields the same answer as above.  The matrix $\left[W_N^{kn}\right]$ is known as the **Fourier Matrix**.

**Q2**: What is the Fourier Matrix $\left[W_8^{kn}\right]$?

**A**: The Fourier Matrix $\left[W_8^{kn}\right]$ is given by:

$$
\left(W_8^{kn}\right)
=
\left(
\begin{array}{cccccccc}
 1 & 1 & 1 & 1 & 1 & 1 & 1 & 1 \\
 1 & e^{-i \pi/4} & -i & e^{- 3 i \pi/4} & -1 & e^{3 i \pi/4 } & i & e^{i \pi/4  } \\
 1 & -i & -1 & i & 1 & -i & -1 & i \\
 1 & e^{-3 i \pi/4} & i & e^{-i \pi/4} & -1 & e^{i \pi/4} & -i & e^{3 i \pi/4} \\
 1 & -1 & 1 & -1 & 1 & -1 & 1 & -1 \\
 1 & e^{3 i \pi/4} & -i & e^{i \pi/4} & -1 & e^{-i \pi /4} & i & e^{-3 i \pi/4} \\
 1 & i & -1 & -i & 1 & i & -1 & -i \\
 1 & e^{i \pi/4} & i & e^{3 i \pi/4} & -1 & e^{-3 i \pi /4} & -i & e^{-i \pi/4} \\
\end{array}
\right)
$$

<a name="11.6"></a>
## 11.6 Properties of the Discrete Fourier Series

<b>Linearity.</b> 

If $a$ and $b$ are real numbers then:

$$a\tilde{x_1}[n]+b\tilde{x_2}[n]\overset{DFS}{\Longleftrightarrow}a\tilde{X_1}[k]+b\tilde{X_2}[k] \tag{37}$$

<b>Shift.</b>

If a periodic sequence $\tilde{x}[n]$ is shifted, the DFS coefficients are multiplied by a complex exponential.  If $\tilde{x}[n]\overset{DFS}{\Longleftrightarrow}\tilde{X}[k]$, then for $\tilde{y}[n] = \tilde{x}[n-n_0]$ we get

$$ \tilde{Y}[k] = W_N^{kn_0} \tilde{X}[k] = e^{-i2\pi k n_0/N}\tilde{X}[k] \tag{38}$$

Similarly if $\tilde{y}[n] = W_N^{nk_0} \tilde{x}[n]$ then

$$ \tilde{Y}[k] = \tilde{X}[k+k_0] \tag{39}$$

<b>Periodic convolution.</b>

If $\tilde{h}[n]$ and $\tilde{x}[n]$ are periodic with period $N$ and DFS coefficients $\tilde{H}[k]$ and $\tilde{X}[k]$, respectively, the sequence 

$$ \tilde{Y}[k]=\tilde{H}[k]\tilde{X}[k] \tag{40}$$

is formed by **periodically convolving** $\tilde{h}[n]$ with $\tilde{x}[n]$

$$\tilde{y}[n] = \sum_{l=0}^{N-1} \tilde{h}[l]\tilde{x}[n-l] \tag{41}$$

which is sometimes written notationally as

$$\tilde{y}[n] = \tilde{h}[n]\ast_N \tilde{x}[n] \tag{42}$$

where the subscript $N$ marks the sum over one period. The only difference between **periodic convolution** and **linear convolution** is that the sum runs over a single period $N$, with the indices of $\tilde{x}$ taken modulo $N$.

<a name="11.7"></a>
## 11.7 The Discrete Fourier Transform

The discrete Fourier Transform (DFT) can easily be developed from the discrete Fourier series (DFS) representation for periodic sequences.  Let $x[n]$ be a finite-length sequence of length $N$ that is equal to zero outside the interval $[0,N-1]$.  A periodic sequence $\tilde{x}[n]$ may be formed from $x[n]$ as follows:

$$\tilde{x}[n] = \sum_{k=-\infty}^{\infty} x[n+kN], \quad n=-\infty,\infty \tag{43}$$

From the inverse DFS representation for $\tilde{x}[n]$, 

$$ \tilde{x}[n] = \frac{1}{N} \sum_{k=0}^{N-1} \tilde{X}[k]e^{i2\pi k n/N}, \quad n=-\infty,\infty \tag{44}$$

we can enforce the following restriction within the range $0\le n < N$. This gives us our representation of the DFT:

<div class="alert alert-info" role="alert">
$$ x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k]e^{i2\pi k n/N},\quad 0\le n < N \tag{45}$$
</div>

Note that it is only by enforcing the range $0\le n < N$ that the DFT is differentiated from the DFS.  Similarly, for the forward DFT representation we can write:

<div class="alert alert-info" role="alert">
$$ X[k] = \sum_{n=0}^{N-1} x[n]e^{-i2\pi k n/N},\quad 0\le k < N \tag{46}$$
</div>

We again recognize the duality with the following notation:

$$x[n] \overset{DFT}{\Longleftrightarrow}X[k] \tag{47}$$

<div class="alert alert-info">
<b>Synthesis &mdash; sampling, now in frequency.</b> In Module 10, sampling $x(t)$ every $T$ made its spectrum periodic. The DFT does the reverse: it keeps only $N$
equally spaced frequencies of the spectrum $\sum_n x[n]e^{-i\omega n T}$ (the DTFT of Section 5.10), and sampling the spectrum makes the time signal periodic
with period $N$, which is the $\tilde{x}[n]$ of equation (43). Every surprise in this module (wrap-around convolution, leakage) is that hidden periodic
extension showing itself.
</div>

<a name="11.7.1"></a>
### 11.7.1 Example 3: four standard DFT pairs

**(Q1)**  Compute the $N$-point DFT of

$$x_1[n] = \delta[n] \quad \mathrm{where} \quad 0\le n < N \tag{48}$$

**A:** The DFT of the unit sample can be easily evaluated from the definition of the DFT:

$$X_1[k] = \sum_{n=0}^{N-1} \delta[n] W_N^{nk} = W_N^{0k} = e^{-i2\pi 0k/N} = 1, \quad 0\le k < N \tag{49}$$

Note that this is similar to the continuous Fourier Transform where $\delta(t)\Longleftrightarrow 1$.  The coefficient and stem plots of the real and imaginary components assuming $N=10$ are shown below:

In [ ]:
N = 10
x3a = np.zeros(N); x3a[0] = 1
X3a = np.fft.fft(x3a); print(np.round(X3a, 6))
stem_ri(X3a, what=r'$X_1[k]$', ylim=(-1.5, 1.5))

<b>Figure 11.4.</b> The real (a) and imaginary (b) parts of the DFT of $\delta[n]$, equation (49), for $N = 10$: $X_1[k] = 1$ for every $k$. <i>(Python-generated.)</i>

**(Q2)** Compute the $N$-point DFT of

$$x_2[n] = \delta[n-n_0], \quad \mathrm{where} \quad 0\le n < N \tag{50}$$

**A:** Similarly to the first example:

$$X_2[k] = \sum_{n=0}^{N-1} \delta[n-n_0] W_N^{nk} = W_N^{n_0k} = e^{-i2\pi n_0 k/N} , \quad 0\le k <N \tag{51}$$

The coefficient and stem plots of the real and imaginary components assuming $N=10$ and $n_0=2$ are shown below which very much look like the outline of a $\cos$ function (which should not be too surprising since we are dealing with complex exponentials).

In [ ]:
N, n0 = 10, 2
x3b = np.zeros(N); x3b[n0] = 1
X3b = np.fft.fft(x3b); print(np.round(X3b, 4))
stem_ri(X3b, what=r'$X_2[k]$', ylim=(-1.3, 1.3))

<b>Figure 11.5.</b> The real (a) and imaginary (b) parts of the DFT of $\delta[n - 2]$, equation (51), for $N = 10$: the samples of $\cos(2\pi\cdot 2k/10)$ and $-\sin(2\pi\cdot 2k/10)$, since $|X_2[k]| = 1$ and the phase falls linearly with $k$. <i>(Python-generated.)</i>

Let's now combine the above two examples into a low-order first derivative filter. However, this time we are going to actually compute the **frequency structure** and plot the magnitude spectra in a more logical order:

The filter $h = [1, -1]$ is the first difference $y[n] = x[n] - x[n-1]$, a discrete derivative. Its DFT follows from the two pairs above and linearity: $H[k] = 1 - W_N^{k}$, so $|H| = 2|\sin(\pi k/N)|$, which grows like $2\pi|f|$ (the response of a true derivative) only at low frequencies.

In [ ]:
C = DSP_COLORS
N = 100
h2 = np.zeros(N); h2[0], h2[1] = 1, -1                                # first difference, 2 taps
h4 = np.zeros(N); h4[:5] = [1/12, -2/3, 0, 2/3, -1/12]                # 4th-order centered stencil, 5 taps
f = np.fft.fftshift(np.fft.fftfreq(N))                                # cycles per sample, -0.5 ... 0.49
H2 = np.abs(np.fft.fftshift(np.fft.fft(h2))); H4 = np.abs(np.fft.fftshift(np.fft.fft(h4)))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw=dict(width_ratios=[1, 1.5]))
ml, sl, bl = axa.stem(np.arange(2), h2[:2], basefmt='k-', label='2-tap $[1, -1]$')
plt.setp(sl, color=C['impulse_resp']); plt.setp(ml, color=C['impulse_resp'], ms=7)
ml, sl, bl = axa.stem(np.arange(5) + 0.15, h4[:5], basefmt='k-', label='5-tap, 4th order')
plt.setp(sl, color=C['phase']); plt.setp(ml, color=C['phase'], ms=5, marker='s')
axa.set_xlabel('sample index $n$'); axa.set_ylabel('$h[n]$'); axa.set_title('(a) Filter coefficients')
axa.set_xlim(-0.6, 4.8); axa.legend(loc='lower right', fontsize=9)
ff = np.linspace(-0.5, 0.5, 501)
axb.plot(ff, 2*np.pi*np.abs(ff), '--', color=C['neutral'], lw=1.5, label=r'ideal derivative $2\pi|f|$')
axb.plot(ff, 2*np.abs(np.sin(np.pi*ff)), color=C['freq_domain'], lw=1.5, label=r'2-tap: $2|\sin\pi f|$')
axb.plot(f, H2, 'o', color=C['freq_domain'], ms=3.5, label='2-tap: DFT, $N = 100$')
axb.plot(f, H4, color=C['phase'], lw=1.5, label='5-tap: DFT, $N = 100$')
axb.set_xlim(-0.52, 0.52); axb.set_ylim(0, 3.3)
axb.set_xlabel('frequency $f$ [cycles per sample]'); axb.set_ylabel('$|H(f)|$')
axb.set_title('(b) Magnitude response'); axb.legend(loc='upper center', fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

<b>Figure 11.6.</b> The first-difference filter as a derivative. (a) Coefficients of the 2-tap filter $[1, -1]$ (green) and of a 5-tap, fourth-order stencil $[1/12, -2/3, 0, 2/3, -1/12]$ (purple). (b) Magnitude responses from 100-point DFTs (dots and purple curve) against the analytic $2|\sin\pi f|$ (red) and the ideal derivative $2\pi|f|$ (dashed). Both follow the ideal line at low frequency, where the longer stencil is more accurate (at $f = 0.1$ the errors are -1.6% and -0.5%). At high frequency both fall below it: the 2-tap response rises to 2 at the Nyquist frequency $|f| = 0.5$, while the 5-tap response peaks at 1.37 and returns to zero there. <i>(Python-generated.)</i>

**(Q3)** Compute the $N$-point DFT of

$$x_3[n] = \alpha^n, \quad 0\le n < N \tag{52}$$

**A:** The DFT is as follows:

$$X_3[k] = \sum_{n=0}^{N-1} x[n] W_N^{nk} =\sum_{n=0}^{N-1} \alpha^n W_N^{nk}=\sum_{n=0}^{N-1} \left(\alpha W_N^{k}\right)^n , \quad \quad 0\le k < N \tag{53}$$

 Again, using the geometric sum of equation (26):

$$X_3[k] = \frac{1- \left(\alpha W_N^{k}\right)^N}{1-\alpha W_N^{k}}, \quad 0\le k < N \tag{54}$$

The coefficient and stem plots of the real and imaginary components for $N=10$ and $\alpha=0.8$ are shown below.

Because $W_N^{kN} = e^{-i2\pi k} = 1$, the numerator simplifies to $1 - \alpha^N$, so $X_3[k] = (1 - \alpha^N)/(1 - \alpha W_N^{k})$.

In [ ]:
N, a = 10, 0.8
n = np.arange(N)
x3c = a**n
X3c = np.fft.fft(x3c); print(np.round(X3c, 4))
stem_ri(X3c, what=r'$X_3[k]$', ylim=(-2.3, 4.6))

<b>Figure 11.7.</b> The real (a) and imaginary (b) parts of the DFT of $\alpha^n$, equation (54), for $N = 10$ and $\alpha = 0.8$. <i>(Python-generated.)</i>

**(Q4)** Compute the $N$-point DFT of

$$x_4[n] = u[n] - u[n-n_0], \quad 0\le n < N \tag{55}$$

**A:** The DFT may be calculated directly

$$X_4[k] = \sum_{n=0}^{N-1} x[n] W_N^{nk} = \sum_{n=0}^{n_0-1} W_N^{nk} = \frac{1- \left(W_N^{k}\right)^{n_0}}{1-W_N^{k}} = \frac{1- W_N^{k n_0}}{1-W_N^{k}} , \quad 0\le k < N \tag{56}$$

where we have used $r=W_N^{k}$. The coefficient and stem plots of the real and imaginary components are shown below for $N=10$ and $n_0=5$.

In [ ]:
N, n0 = 10, 5
x3d = np.ones(N); x3d[n0:N] = 0
X3d = np.fft.fft(x3d); print(np.round(X3d, 4))
stem_ri(X3d, what=r'$X_4[k]$', ylim=(-3.5, 5.5))

<b>Figure 11.8.</b> The real (a) and imaginary (b) parts of the DFT of $u[n] - u[n - n_0]$, equation (56), for $N = 10$ and $n_0 = 5$. <i>(Python-generated.)</i>

<a name="11.7.2"></a>
### 11.7.2 Example 4: an inverse DFT

**Q:** Find the inverse DFT of the $N=10$ point signal

$$X[k] = 
\left\{
\begin{array}{cc}
3, & k=0\\
1, & 1\le k \le 9\\
\end{array}
\right.{}
\tag{57}
$$

**A:** To find the inverse DFT, we note that $X[k]$ may exploit the linearity property of the DFT and write:

$$X[k] = X_1[k]+X_2[k] = 1+2\delta[k] \tag{58}$$

From the example above we know that 

$$\delta[n] \overset{DFT}{\Longleftrightarrow} 1 \tag{59}$$

and

$$1\overset{DFT}{\Longleftrightarrow} N\delta[k] \tag{60}$$

Thus, 

$$\delta[n]+\frac{1}{5} \overset{DFT}{\Longleftrightarrow} 1+2\delta[k] \tag{61}$$

The coefficients and stem plots of the real and imaginary components are shown below.

In [ ]:
X2 = np.array([3, 1, 1, 1, 1, 1, 1, 1, 1, 1])
x2 = np.fft.ifft(X2); print(np.round(x2, 6))
stem_ri(x2, xlabel='sample index $n$', ylabel='amplitude', what='$x[n]$', ylim=(-0.3, 1.4))

<b>Figure 11.9.</b> The inverse DFT of Example 4, equation (61): (a) $x[n] = \delta[n] + 1/5$, i.e. $[1.2, 0.2, \ldots, 0.2]$; (b) the imaginary part is zero. <i>(Python-generated.)</i>

<a name="11.7.3"></a>
### 11.7.3 Example 5: a squared cosine

**Q:** Find the N-point DFT of:

$$x[n] = 4+\mathrm{cos}^2\left(\frac{2\pi n}{N}\right), \quad n=0,N-1 \tag{62}$$

**A:** The DFT of this sequence may be evaluated by expanding cosine as a sum of complex exponentials:

$$x[n] = 4 + \left( 
\frac{e^{i2\pi n/N} + e^{-i2\pi n/N}}{2}
\right)^2 
=
4 + \frac{1}{4}\left(
e^{i4\pi n/N} + 2 + e^{-i4\pi n/N}
\right)
=
\frac{9}{2} + \frac{1}{4} e^{i4\pi n/N} + \frac{1}{4} e^{-i4\pi n/N}
\tag{63}
$$

Using the periodicity we rewrite the last term as:

$$x[n] =\frac{9}{2} + \frac{1}{4} e^{\frac{i2\pi}{N} (2n)} + \frac{1}{4} e^{\frac{i2\pi}{N} (N-2)n}
\tag{64}
$$

Thus, writing

$$ X[k] = \sum_{n=0}^{N-1} \left( \frac{9}{2} + \frac{1}{4} e^{\frac{i2\pi}{N} (2n)} + \frac{1}{4} e^{\frac{i2\pi}{N} (N-2)n}\right) e^{-\frac{i2\pi k}{N} n},\quad 0\le k < N
\tag{65}
$$

Expressing these as three sums:

$$ X[k] = 
\frac{9}{2} 
\sum_{n=0}^{N-1} 
e^{-\frac{i2\pi k}{N} n} 
+ 
\frac{1}{4} \sum_{n=0}^{N-1} 
e^{\frac{i2\pi}{N} (2n)} e^{-\frac{i2\pi k}{N} n} 
+ 
\frac{1}{4}\sum_{n=0}^{N-1} 
 e^{\frac{i2\pi}{N} (N-2)n} e^{-\frac{i2\pi k}{N} n}
,\quad 0\le k < N
\tag{66}
$$

Combining the exponential terms yields:

$$ X[k] = 
\frac{9}{2} 
\sum_{n=0}^{N-1} 
e^{-\frac{i2\pi k}{N} n} 
+ 
\frac{1}{4} \sum_{n=0}^{N-1} 
e^{\frac{-i2\pi (k-2)}{N} n} 
+ 
\frac{1}{4}\sum_{n=0}^{N-1} 
 e^{\frac{i2\pi}{N} (N-2-k)n}
,\quad 0\le k < N
\tag{67}
$$
or evaluating for different $k$ values:

$$ X[k] = \left\{
\begin{array}{cc}
\frac{9N}{2}, & k=0\\
\frac{N}{4}, & k=2, N-2\\
0 , & \mathrm{otherwise}\\
\end{array}
\right.{}
\tag{68}
$$

The coefficient and stem plots of the real and imaginary components are shown below:

In [ ]:
N = 10
n = np.arange(N)
x4 = 4 + np.cos(2*np.pi*n/N)**2
X4 = np.fft.fft(x4); print(np.round(X4, 6))
stem_ri(X4, what='$X[k]$', ylim=(-2, 9*N/2 + 2))

<b>Figure 11.10.</b> The real (a) and imaginary (b) parts of the DFT of $4 + \cos^2(2\pi n/N)$, equation (68), for $N = 10$: $X[0] = 9N/2 = 45$ and $X[2] = X[8] = N/4 = 2.5$. <i>(Python-generated.)</i>

<a name="11.8"></a>
## 11.8 The DFT as a Matrix

Recall from above that we can represent the DFS as a matrix operation.  Due to the similarity between the DFS and DFT operations, the same can be said for the DFT:

$$
\left(
\begin{array}{c}
\tilde{X}[0] \\
\tilde{X}[1]\\
\tilde{X}[2]\\
\dots \\
\tilde{X}[N-3]\\
\tilde{X}[N-2]\\
\tilde{X}[N-1]\\
\end{array}
\right)
=
\left(
\begin{array}{c}
W_N^{0*0} & W_N^{1*0} & W_N^{2*0} & \vdots & W_N^{(N-3)*0} & W_N^{(N-2)*0} & W_N^{(N-1)*0} \\
W_N^{0*1} & W_N^{1*1} & W_N^{2*1} & \vdots & W_N^{(N-3)*1} & W_N^{(N-2)*1} & W_N^{(N-1)*1} \\
W_N^{0*2} & W_N^{1*2} & W_N^{2*2} & \vdots & W_N^{(N-3)*2} & W_N^{(N-2)*2} & W_N^{(N-1)*2} \\
\dots     & \dots     & \dots     & \vdots  & \dots     & \dots     & \dots     \\
W_N^{0*(N-3)} & W_N^{1*(N-3)} & W_N^{2*(N-3)} & \vdots & W_N^{(N-3)*(N-3)} & W_N^{(N-2)*(N-3)} & W_N^{(N-1)*(N-3)} \\
W_N^{0*(N-2)} & W_N^{1*(N-2)} & W_N^{2*(N-2)} & \vdots & W_N^{(N-3)*(N-2)} & W_N^{(N-2)*(N-2)} & W_N^{(N-1)*(N-2)} \\
W_N^{0*(N-1)} & W_N^{1*(N-1)} & W_N^{2*(N-1)} & \vdots & W_N^{(N-3)*(N-1)} & W_N^{(N-2)*(N-1)} & W_N^{(N-1)*(N-1)} \\
\end{array}
\right)
\left(
\begin{array}{c}
\tilde{x}[0] \\
\tilde{x}[1] \\
\tilde{x}[2] \\
\dots \\
\tilde{x}[N-3] \\
\tilde{x}[N-2] \\
\tilde{x}[N-1] \\
\end{array}
\right) 
\tag{69}
$$

The columns of this matrix are orthogonal, by equation (10), so

$$\mathbf{F}^H\mathbf{F} = N\,\mathbf{I}, \qquad \mathbf{F}^{-1} = \frac{1}{N}\mathbf{F}^H \tag{70}$$

where $\mathbf{F} = [W_N^{kn}]$ and $^H$ is the conjugate transpose. The inverse DFT, equation (45), is exactly this inverse. The cell below checks it numerically.

In [ ]:
# The DFT matrix F has orthogonal columns: F^H F = N I, so every singular value equals sqrt(N)
for N in (4, 8, 64):
    F = np.fft.fft(np.eye(N))                       # column j is the DFT of the j-th unit vector
    s = np.linalg.svd(F, compute_uv=False)
    err = np.max(np.abs(F.conj().T @ F - N*np.eye(N)))
    print(f'N = {N:2d}:  max|F^H F - N I| = {err:.1e};  all singular values = sqrt(N) = {np.sqrt(N):.3f}: {np.allclose(s, np.sqrt(N))}')

<div class="alert alert-info">
<b>Synthesis &mdash; the DFT is a matrix, and a very special one.</b> In Module 8 (Section 8.4) every linear operation on a sampled signal became a
matrix, and the SVD described it as rotate&ndash;stretch&ndash;rotate. The DFT matrix $\mathbf{F}$ has orthogonal columns, the complex exponentials of
equation (10), and $\mathbf{F}^H\mathbf{F} = N\mathbf{I}$, so all of its singular values equal $\sqrt{N}$: $\mathbf{F}$ rotates and scales uniformly,
and it neither stretches nor loses any direction. These columns are the circulant singular vectors that Module 8's parting thought (Section 8.8) pointed to,
and they are why the inverse DFT is simply $\mathbf{F}^H/N$.
</div>

<a name="11.9"></a>
## 11.9 Properties of the DFT

Here I list some of the important properties of the DFT.  Because each sequence is assumed to be finite in length, one must be careful manipulating DFTs to avoid unintended behavior (e.g., aliasing).

<b>Linearity.</b>

If $x_1[n]$ and $x_2[n]$ have N-point DFTs $X_1[k]$ and $X_2[k]$, respectively, then 

$$ax_1[n]+bx_2[n] \overset{DFT}{\Longleftrightarrow}aX_1[k]+bX_2[k] \tag{71}$$

NOTE: When using this property it is important to ensure that the DFTs are the same length.  If they are unequal, then the shorter sequence must be **padded** with zeros to make it the same length as the longer one. 

<b>Symmetry.</b>

If $x[n]$ is real valued then $X[k]$ is conjugate symmetric

$$X[k] = \overline{X[-k]} = \overline{X[N-k]} \tag{72}$$


The second form is what you see in FFT output: for a real signal, bin $N - k$ holds the complex conjugate of bin $k$, so the upper half of the array
(the negative frequencies, Section 11.12) carries no new information.

<b>Shift.</b> As for the DFS, by equation (38), a circular shift by $n_0$ samples multiplies the DFT by $W_N^{kn_0}$: $x[(n - n_0) \bmod N] \overset{DFT}{\Longleftrightarrow} W_N^{kn_0}X[k]$.

<a name="11.10"></a>
## 11.10 Linear Convolution with the DFT

The DFT provides a convenient way to perform convolutions without having to evaluate the convolution sum.  If $h[n]$ is $N_1$ points long and $x[n]$ is $N_2$ points long, then $h[n]$ may be linearly convolved with $x[n]$ according to the following formula:

1. Pad the sequences $h[n]$ and $x[n]$ with zeros so that they are both of length $N\ge N_1+N_2-1$.
2. Find the $N$-point DFTs of $h[n]$ and $x[n]$.
3. Multiply the two DFTs to form the product $Y[k] = H[k] X[k]$.
4. Find the inverse DFT of $Y[k]$.

<div class="alert alert-info">
<b>Synthesis &mdash; circular vs linear convolution.</b> Module 9 (Section 9.7) computed $y = h \ast x$ with a finite sum, and Module 7 (Section 7.4) showed that
convolution becomes multiplication of spectra. The DFT keeps that promise only for <i>periodic</i> signals: $H[k]X[k]$ is the periodic convolution of
equation (41), so whatever spills past sample $N - 1$ wraps around to the start. Zero padding to $N \ge N_1 + N_2 - 1$ (step 1 above) leaves room for
the spill, and the circular result then equals the linear one. In Module 8's language, the DFT diagonalizes the circulant matrix, not the Toeplitz one.
</div>

In [ ]:
C = DSP_COLORS
x = np.array([3, 1, 0, 0, 2.]); h = np.array([1, 0.5, 0.25])
lin = np.convolve(x, h)                                               # length 5 + 3 - 1 = 7
circ = np.real(np.fft.ifft(np.fft.fft(x)*np.fft.fft(h, 5)))           # N = 5: no room for the spill
pad = np.real(np.fft.ifft(np.fft.fft(x, 7)*np.fft.fft(h, 7)))         # N = 7: zero padded
def stems(ax, n, v, col, **kw):
    ml, sl, bl = ax.stem(n, v, basefmt='k-', **kw)
    plt.setp(sl, color=col); plt.setp(ml, color=col, ms=6)
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
stems(axs[0], np.arange(5), x, C['input'], label='$x[n]$')
stems(axs[0], np.arange(3) + 0.15, h, C['impulse_resp'], label='$h[n]$')
axs[0].set_title('(a) Input and filter'); axs[0].legend(fontsize=10)
stems(axs[1], np.arange(5), lin[:5], C['output'], label='samples 0-4 of $h \\ast x$')
axs[1].bar([0, 1], lin[5:], bottom=lin[:2], width=0.35, color=C['alias'], alpha=0.8, label='wrapped from $n = 5, 6$')
axs[1].plot(np.arange(5), circ, 'o', mfc='none', mec='k', ms=11, mew=1.5, label='circular result, $N = 5$')
axs[1].set_title('(b) 5-point DFTs: circular'); axs[1].legend(fontsize=9, loc='upper right')
stems(axs[2], np.arange(7), pad, C['output'], label='7-point DFTs (padded)')
axs[2].plot(np.arange(5, 7), lin[5:], 's', color=C['alias'], ms=9, mfc='none', mew=2, label='samples that wrap in (b)')
axs[2].set_title('(c) 7-point DFTs: linear'); axs[2].legend(fontsize=9, loc='upper right')
for ax in axs:
    ax.set_xlabel('sample index $n$'); ax.set_xlim(-0.6, 6.6); ax.set_ylim(0, 5.2)
axs[0].set_ylabel('amplitude')
plt.tight_layout(); plt.show()
print('linear (N = 7):  ', np.round(pad, 4))
print('circular (N = 5):', np.round(circ, 4))

<b>Figure 11.11.</b> Why step 1 matters. (a) $x[n] = [3, 1, 0, 0, 2]$ (blue) and $h[n] = [1, 0.5, 0.25]$ (green). (b) With 5-point DFTs the product $H[k]X[k]$ gives the circular result (black circles): the last two samples of the linear convolution, $1.0$ and $0.5$ at $n = 5, 6$ (orange), wrap onto $n = 0, 1$, giving $4$ and $3$ instead of $3$ and $2.5$. (c) Zero padding to $N = 7 = 5 + 3 - 1$ reproduces the linear convolution exactly; the orange squares mark the samples that wrapped in (b). <i>(Python-generated.)</i>

The recipe is also fast. Figure 11.12 times it against `np.convolve`, which evaluates the convolution sum directly.

In [ ]:
import time
from scipy import signal
C = DSP_COLORS

def fft_recipe(x, h):
    """Steps 1-4: pad both to N >= N1 + N2 - 1, transform, multiply, inverse transform."""
    N = len(x) + len(h) - 1
    return np.fft.ifft(np.fft.fft(x, N)*np.fft.fft(h, N)).real

def median_time(fun, x, h, reps=5):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fun(x, h); ts.append(time.perf_counter() - t0)
    return np.median(ts)

lengths = np.unique(np.r_[np.round(np.geomspace(16, 8192, 26)).astype(int), 4099, 8191])   # incl. two primes
t_direct, t_recipe, t_scipy = [], [], []
for L in lengths:
    x = np.exp(-np.arange(L)/50.0)
    h = np.zeros(L); h[0], h[L//2] = 1, -1
    t_direct.append(median_time(np.convolve, x, h))
    t_recipe.append(median_time(fft_recipe, x, h))
    t_scipy.append(median_time(lambda a, b: signal.fftconvolve(a, b, mode='full'), x, h))
t_direct, t_recipe, t_scipy = map(np.array, (t_direct, t_recipe, t_scipy))

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.5))
axa.loglog(lengths, t_direct, 'o-', color=C['neutral'], ms=4, label='np.convolve (direct sum)')
axa.loglog(lengths, t_recipe, 's-', color=C['freq_domain'], ms=4, label='4-step DFT recipe')
axa.loglog(lengths, t_scipy, '^-', color=C['time_domain'], ms=4, label='signal.fftconvolve')
ref = lengths.astype(float)
axa.loglog(ref, t_direct[-1]*(ref/ref[-1])**2, ':', color='k', lw=1, label=r'slope $N^2$')
axa.loglog(ref, t_recipe[-1]*(ref*np.log2(ref))/(ref[-1]*np.log2(ref[-1])), '--', color='k', lw=1, label=r'slope $N\log_2 N$')
axa.set_xlabel('length of $x[n]$ and $h[n]$'); axa.set_ylabel('run time [s]'); axa.set_title('(a) Run time (median of 5)')
axa.legend(fontsize=8, loc='upper left')
axb.semilogx(lengths, t_direct/t_recipe, 's-', color=C['freq_domain'], ms=4, label='4-step DFT recipe')
axb.semilogx(lengths, t_direct/t_scipy, '^-', color=C['time_domain'], ms=4, label='signal.fftconvolve')
axb.axhline(1, color='k', lw=1)
axb.set_xlabel('length of $x[n]$ and $h[n]$'); axb.set_ylabel('speed-up over np.convolve'); axb.set_title('(b) Speed-up')
axb.legend(fontsize=9, loc='upper left')
plt.tight_layout(); plt.show()

<b>Figure 11.12.</b> Cost of convolving two sequences of equal length $N$, from 16 to 8192 samples (on a logarithmic grid that includes the primes 4099 and 8191). (a) Median run time of the direct sum (`np.convolve`), the four-step DFT recipe, and `scipy.signal.fftconvolve`, with reference slopes $N^2$ and $N\log_2 N$. (b) Speed-up of the two FFT methods over the direct sum. The timings depend on the computer and change slightly every time the cell is run, so no number is quoted from them. <i>(Python-generated.)</i>

<a name="11.11"></a>
## 11.11 Fast Fourier Transforms

You may be surprised by how uneven the timings in Figure 11.12 are. This is partially because of something we've only briefly talked about: the [fast Fourier Transform (FFT)](https://en.wikipedia.org/wiki/Fast_Fourier_transform).  When the length of the arrays can be factorized by powers of 2,3,5,7 or 11, a FFT algorithm can be used to split up the full DFT into a number of sub-FFTs that are smaller and more efficient to compute.  If the resulting lengths are again divisible by a power of 2,3,5,7 and 11, then it can be again split. 

For a sequence of length $N = 2^m$, where $m$ is an integer, the DFT can be split in half repeatedly down to 2-point DFTs, and the results recombined through $m = \log_2 N$ stages to give the full $N$-point FFT. Each stage costs about $N$ operations, so the FFT needs $\mathcal{O}(N \log_2 N)$ operations, while the DFT computed as a matrix&ndash;vector product (Section 11.8) needs $\mathcal{O}(N^2)$.  Thus, it makes sense to pad a few zeros to a length that is optimal for FFT!

<a name="11.12"></a>
## 11.12 The Frequency Axis, Resolution and Leakage

So far the examples have used the frequency <i>index</i> $k$. A recorded trace has a sample interval $\Delta t$, and bin $k$ of an $N$-point DFT
belongs to the frequency

$$f_k = \frac{k}{N\Delta t}, \qquad k = 0, 1, \ldots, N-1 \tag{73}$$

Because the DFT is periodic in $k$, by equation (19), the bins with $k > N/2$ are the same as the bins $k - N$: they hold the <i>negative</i> frequencies.
`np.fft.fftfreq(N, d=dt)` returns the frequencies in exactly this order, and `np.fft.fftshift` moves the negative half to the front for plotting
(Section 5.3.1). The spacing of the bins is

$$\Delta f = \frac{1}{N\Delta t} = \frac{1}{T} \tag{74}$$

where $T = N\Delta t$ is the record length. Only the record length, not the sample interval, sets how finely the DFT divides the frequency axis.

<div class="alert alert-info">
<b>Note:</b> Call <code>np.fft.fftfreq(N, d=dt)</code> with the sample interval. Without <code>d</code>, the frequencies come out in cycles per sample
(between $-0.5$ and $0.5$), not in hertz.
</div>

<div class="alert alert-info">
<b>Synthesis &mdash; resolution is set by record length.</b> $\Delta f = 1/T$ in equation (74) is Module 5's scaling theorem in discrete form: to
narrow a spectral line you must lengthen the record. Adding zeros to the end of the record (Figure 11.13) changes $N$ but not $T$, so it draws the same
smooth spectrum on a finer grid without separating anything new. Module 10's section on practical Fourier sampling made the same point from the sampling side.
</div>

What, then, are the DFT bins samples <i>of</i>? Comparing equation (46) with the DTFT of Section 5.10, $\widehat{X_s}(\omega) = \sum_n x[n]e^{-i\omega n\Delta t}$,
for a record of $N$ samples:

$$X[k] = \widehat{X_s}(\omega)\Big|_{\omega = 2\pi f_k} \tag{75}$$

The DFT samples the continuous spectrum of the finite record at $N$ equally spaced frequencies. Zero padding to length $8N$ evaluates the same
spectrum at eight times as many frequencies, as Figure 11.13 shows for two tones only 0.3 Hz apart.

In [ ]:
C = DSP_COLORS
dt = 0.01                                        # 100 samples per second
def spectrum(T, pad):
    n = np.arange(int(round(T/dt)))
    x = np.cos(2*np.pi*10.0*n*dt) + np.cos(2*np.pi*10.3*n*dt)      # two tones 0.3 Hz apart
    Np = pad*len(n)
    return np.fft.rfftfreq(Np, dt), np.abs(np.fft.rfft(x, Np))*2/len(n)
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, T, ttl in ((axs[0], 2.0, r'(a) $T = 2$ s, $\Delta f = 0.5$ Hz'), (axs[1], 5.0, r'(b) $T = 5$ s, $\Delta f = 0.2$ Hz')):
    f8, A8 = spectrum(T, 8); f1, A1 = spectrum(T, 1)
    ax.plot(f8, A8, color=C['continuous'], lw=1.5, label='zero padded to $8N$')
    ml, sl, bl = ax.stem(f1, A1, basefmt=' ', label='unpadded DFT bins')
    plt.setp(sl, color=C['freq_domain']); plt.setp(ml, color=C['freq_domain'], ms=5)
    for f0 in (10.0, 10.3):
        ax.axvline(f0, ls=':', color=C['time_domain'], lw=1)
    ax.set_xlim(8, 12.5); ax.set_ylim(0, 2.3); ax.set_title(ttl); ax.set_xlabel('frequency [Hz]')
axs[0].set_ylabel('amplitude'); axs[0].legend(fontsize=9, loc='upper left')
plt.tight_layout(); plt.show()

<b>Figure 11.13.</b> Zero padding interpolates but does not resolve. Two equal tones at 10.0 and 10.3 Hz (dotted lines), sampled every 0.01 s. (a) A 2 s record ($\Delta f$ = 0.5 Hz): the unpadded DFT bins (red) and the spectrum zero padded to $8N$ (gray) both show a single peak (at 10.0 Hz unpadded, 10.19 Hz padded). (b) A 5 s record ($\Delta f$ = 0.2 Hz): the padded spectrum separates the two tones (peaks at 10.0 and 10.3 Hz), because the record is longer than $1/(0.3\ \mathrm{Hz}) = 3.3$ s. <i>(Python-generated.)</i>

<div class="alert alert-warning">
<b>Common misconception:</b> "Zero padding improves frequency resolution." Padding draws the spectrum of the same record on a finer grid. The resolution, the
ability to separate two close frequencies, is set by the record length $T$ through equation (74); compare panels (a) and (b) of Figure 11.13.
</div>

The DFT also has a subtler habit. Equation (10) says that a complex exponential that completes a whole number $k_0$ of cycles in the record
lands in a single bin. If it completes $k_0 = 4.5$ cycles, the geometric series of equation (26) no longer collapses: for $x[n] = e^{i2\pi k_0 n/N}$,

$$|X[k]| = \left|\frac{\sin\big(\pi(k_0 - k)\big)}{\sin\big(\pi(k_0 - k)/N\big)}\right| \tag{76}$$

which is nonzero in every bin. The energy is still all there (Parseval's theorem holds exactly), but it has <b>leaked</b> out of the bin where the tone
belongs. The cause is the hidden periodic extension: a tone that does not fit the record jumps at the wrap from sample $N - 1$ back to sample 0, and a
jump needs many frequencies to represent it.

The animation in Figure 11.14 slides a cosine from exactly 4 cycles per record to 5.45 cycles in steps of 0.05 (30 frames), with $N = 32$. Watch
three things. In the top panel, the gray copy is what the DFT assumes comes next: at 4.00 cycles it continues the record smoothly, and at any other
frequency there is a kink at the dotted line. In the bottom panel, the single red stem at $k = 4$ (height 16) splits into many stems as soon as the tone
moves off the bin; at 4.50 cycles (frame 11) the two nearest bins read only 9.78 and 10.60, and 15.3% of the energy lies
outside bins 3&ndash;5. At exactly 5.00 cycles (frame 21) the energy collapses back into a single stem at $k = 5$. Finally, the red stems always sit on the gray curve: the DFT bins are samples of the zero-padded spectrum, equation (75).

In [ ]:
C = DSP_COLORS
N, Npad = 32, 1024
n = np.arange(N)
k0s = 4 + 0.05*np.arange(30)                     # 4.00 ... 5.45 cycles per record; j = 10 is 4.5, j = 20 is on-bin 5.0
fk = np.arange(Npad//2)*N/Npad                   # fine frequency axis in units of bins
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.5))
state = {}

def update(j):
    k0 = k0s[j]
    x = np.cos(2*np.pi*k0*n/N)
    X = np.fft.fft(x); Xp = np.fft.fft(x, Npad)
    state.update(k0=k0, x=x, X=X)
    a1.cla(); a2.cla()
    for idx, col, lab in ((n, C['sampled'], 'the record, $n = 0 \\ldots 31$'), (n + N, C['neutral'], 'its implied periodic repeat')):
        ml, sl, bl = a1.stem(idx, x, basefmt='k-', label=lab)
        plt.setp(sl, color=col); plt.setp(ml, color=col, ms=4)
    a1.axvline(N - 0.5, color='k', ls=':', lw=1.2)
    a1.set_xlim(-1, 2*N); a1.set_ylim(-1.25, 1.6); a1.set_xlabel('sample $n$'); a1.set_ylabel('$x[n]$')
    a1.legend(loc='upper right', fontsize=9, ncol=2)
    a2.plot(fk, np.abs(Xp[:Npad//2]), color=C['continuous'], lw=1, label='zero-padded spectrum (1024 points)')
    ml, sl, bl = a2.stem(n[:N//2 + 1], np.abs(X[:N//2 + 1]), basefmt='k-', label='$|X[k]|$, 32-point DFT')
    plt.setp(sl, color=C['freq_domain']); plt.setp(ml, color=C['freq_domain'], ms=5)
    a2.set_xlim(-0.5, N/2 + 0.5); a2.set_ylim(0, 18.5)
    a2.set_xlabel('frequency index $k$ [cycles per record]'); a2.set_ylabel('$|X[k]|$'); a2.legend(loc='upper right', fontsize=9)
    fig.suptitle(f'{k0:.2f} cycles per record:  peak $|X[k]|$ = {np.abs(X[:N//2]).max():.1f}  (on-bin value {N//2})', fontsize=13)
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=len(k0s), interval=350, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 11.14.</b> Spectral leakage, 30 frames. A cosine completing 4.00 to 5.45 cycles in an $N = 32$ record. (a) The record (blue) and its implied periodic repeat (gray); the dotted line is the wrap. (b) $|X[k]|$ from the 32-point DFT (red) on the zero-padded spectrum (gray). On-bin, all the energy is in $k = 4$; at 4.25 cycles the peak drops to 14.77, and at 4.50 cycles to 10.60, 3.6 dB below the on-bin value of 16. <i>(Python-generated.)</i>

<a name="11.13"></a>
## 11.13 Parting Thoughts

The DFT is the Fourier series of a periodic sequence, applied to one period. Everything distinctive about it follows from that: $N$ samples give $N$
frequencies spaced $1/T$ apart; products of DFTs are periodic convolutions; and frequencies that do not fit the record leak into every bin. It is
also exact (a unitary matrix, up to the factor $\sqrt{N}$), and the FFT computes it in $\mathcal{O}(N\log_2 N)$ operations. Choosing a taper that trades
leakage against resolution is the subject of Module 12.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 11.1 &mdash; Resolving two close frequencies</b>

A seismometer trace is recorded for $T = 4$ s with $\Delta t = 2$ ms. It contains two lightly damped resonances at 30.00 Hz and 30.15 Hz of equal amplitude.

<b>(a)</b> Find $N$, the frequency spacing $\Delta f$ of the DFT and the Nyquist frequency. At which index $k$ does +60 Hz appear in the output of <code>np.fft.fft</code>, and at which index does −60 Hz appear?

<b>(b)</b> Express the separation of the two resonances in DFT bins. Can the 4 s record resolve them? Two equal tones with frequency difference $\delta f$ beat with period $1/\delta f$ (PHGN 200): compare that period with the record length needed for $\Delta f \le \delta f$, and explain the coincidence.

<b>(c)</b> A colleague zero-pads the 4 s trace to 8192 samples, which gives a bin spacing of 0.061 Hz, and claims that the two peaks are now resolved. What will the padded spectrum actually show near 30 Hz, and what would resolve them?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 11.1</b></summary>

<b>(a)</b> $N = T/\Delta t = 4/0.002 = 2000$. $\Delta f = 1/(N\Delta t) = 1/T = 0.25$ Hz (equation 74). $f_N = 1/(2\Delta t) = 250$ Hz. +60 Hz is at $k = 60/0.25 = 240$; −60 Hz is at $k = N - 240 = 1760$, because the second half of the FFT output holds the negative frequencies (Section 11.12).

<b>(b)</b> $\delta f = 0.15$ Hz $= 0.15/0.25 = 0.6$ bins, less than one bin, so the resonances are not resolved. Resolution needs $\Delta f = 1/T \le \delta f$, i.e. $T \ge 1/0.15 = 6.67$ s. The beat period is also $1/\delta f = 6.67$ s. This is no coincidence: to tell two tones apart you must record long enough to see them drift one full cycle out of step, which is one beat. A shorter record cannot distinguish the pair from a single tone with a slowly varying amplitude (the beat pattern of Module 5).

<b>(c)</b> The padded spectrum is a finer sampling of the <i>same</i> 4 s spectrum (equation 75), so it shows one smooth peak (at 30.09 Hz for two pure tones). Padding changes $N$, not $T$ (Figure 11.13). Only a longer record resolves the pair: an 8 s record gives two peaks, at 29.99 and 30.16 Hz.
</details>

<div class="alert alert-warning">
<b>Microquestion 11.2 &mdash; When the FFT convolves the wrong thing</b>

A two-layer reflectivity series $r = [1, 0, 0, 0, 0, -0.5]$ (6 samples) is convolved with the short wavelet $w = [1, 2, 1]$ to make a synthetic seismogram, as in the convolutional model of Module 7 (Section 7.1).

<b>(a)</b> Compute the linear convolution $y = w \ast r$. How long is it, and what is the minimum DFT length that reproduces it exactly?

<b>(b)</b> Now compute $y$ with 6-point DFTs (no padding), i.e. the periodic convolution of equation (41). Write out the result and explain sample by sample how it is related to the linear result. What happened to the first reflection?

<b>(c)</b> "FFT convolution is always faster than direct convolution, so always use it." Critique this using the operation counts for a 3-point filter applied to a $10^6$-sample trace (direct: about $N M$ multiply&ndash;adds; one FFT of length $2^{20}$: about $2^{20}\times 20$ operations, and three are needed). MATH 348 gives the convolution theorem; what does it not tell you?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 11.2</b></summary>

<b>(a)</b> $y = [1, 2, 1, 0, 0, -0.5, -1, -0.5]$, of length $6 + 3 - 1 = 8$, so $N \ge 8$ (any $N \ge 8$ with zero padding gives exactly this).

<b>(b)</b> With $N = 6$ the samples that would fall at $n = 6, 7$ wrap to $n = 0, 1$: $y_c[0] = y[0] + y[6] = 1 - 1 = 0$, $y_c[1] = y[1] + y[7] = 2 - 0.5 = 1.5$, and the rest are unchanged: $y_c = [0, 1.5, 1, 0, 0, -0.5]$. The tail of the deep reflection has wrapped onto the shallow one and cancelled its peak (compare Figure 11.11). A periodic convolution treats the record as one period of a repeating signal (Section 11.7), so energy that leaves the end comes back at the start.

<b>(c)</b> Direct: $10^6 \times 3 = 3\times 10^6$ multiply&ndash;adds. FFT route: three transforms of length $2^{20} \ge N + M - 1$, about $3 \times 2^{20} \times 20 \approx 6.29e+07$ operations, roughly 20 times more. The convolution theorem says the two routes give the same answer, not which is cheaper. FFT convolution wins when <i>both</i> sequences are long (its cost grows as $N\log N$ instead of $NM$); for short filters the direct sum is faster. Figure 11.12 shows the advantage growing with length when both sequences are long.
</details>

<div class="alert alert-warning">
<b>Microquestion 11.3 &mdash; A tone that does not fit the record</b>

Take $N = 8$ and the complex exponential $x[n] = e^{i2\pi k_0 n/8}$, $n = 0, \ldots, 7$.

<b>(a)</b> For $k_0 = 2$, use orthogonality (equation 10) to write down $X[k]$ for all $k$ without computing any sums.

<b>(b)</b> For $k_0 = 2.5$, sum the geometric series (equation 26, MATH 213) to show $|X[k]| = |\sin(\pi(k_0 - k))|/|\sin(\pi(k_0 - k)/8)|$. Evaluate $|X[2]|$ and $|X[6]|$, and check that $\sum_k|X[k]|^2/N$ equals $\sum_n|x[n]|^2$. How far (in dB) is the largest $|X[k]|$ below the on-bin value of part (a)?

<b>(c)</b> "Leakage shows that the FFT is only an approximation; a more accurate routine (or 128-bit arithmetic) would put all the energy at $k = 2.5$." Explain why this is wrong, using the columns of the DFT matrix (Module 8) and the periodic extension, and name one remedy.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 11.3</b></summary>

<b>(a)</b> $X[k] = \sum_n e^{i2\pi(2 - k)n/8} = 8\delta[k - 2]$: $X[2] = 8$ and every other bin is exactly 0.

<b>(b)</b> With $r = e^{i2\pi(k_0 - k)/8}$, $X[k] = (1 - r^8)/(1 - r)$, and since $|1 - e^{i\theta}| = 2|\sin(\theta/2)|$, $|X[k]| = |\sin(\pi(k_0 - k))|/|\sin(\pi(k_0 - k)/8)|$ (equation 76). For $k_0 = 2.5$ the numerator is 1 for every $k$. $|X[2]| = 1/\sin(\pi/16) = 1/0.1951 = 5.13$, and $|X[3]|$ is the same; $|X[6]| = 1/\sin(3.5\pi/8) = 1.02$. All eight values are $[1.20, 1.80, 5.13, 5.13, 1.80, 1.20, 1.02, 1.02]$, and $\sum|X|^2/8 = 8 = \sum|x|^2$: Parseval holds exactly. The peak is $20\log_{10}(5.13/8) = -3.9$ dB below the on-bin value: no energy has been lost, it has been spread across all bins.

<b>(c)</b> The DFT is exact; nothing is approximated. Its $N$ columns are the only $N$ complex exponentials that complete a whole number of cycles in $N$ samples, and they form an orthogonal basis (Section 11.8; the singular vectors of Module 8). A 2.5-cycle exponential is not one of them, so it must be written as a combination of all eight. Equivalently, the DFT treats the record as one period of a periodic signal (Section 11.7), and the 2.5-cycle tone\'s periodic extension jumps at the wrap; a jump needs many frequencies. Remedies: record a longer or whole-cycle window, or taper the ends of the record toward zero (the taper previewed in Module 10, and the subject of Module 12).
</details>